<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_08_model_tuning/stage_08_02_xgboost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_08_02 - Tuning - XGBoost**

**XGBoost**

- Tuneo grueso

  * `max_depth` → `[3, 5, 7]`
  * `learning_rate` → `[0.01, 0.03, 0.1]`
  * `n_estimators` → `[200, 400, 600]`

- Tuneo fino

  * `subsample` → `[0.6, 0.8, 1.0]`
  * `colsample_bytree` → `[0.6, 0.8, 1.0]`
  * `gamma` → `[0, 0.1, 0.3]`
  * `reg_alpha` → `[0, 0.1, 1]`
  * `reg_lambda` → `[1, 10, 50]`
  * thresholds de probabilidad


# **SETUP COMÚN DEL PIPELINE DE ENTRENAMIENTO**

## **1. Imports**

In [1]:
# Permite anotaciones modernas en Python < 3.11
from __future__ import annotations

# ================================
# Standard library
# ================================
import os
import sys
import json
import time
import random
import importlib
import warnings
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Iterable, Tuple

# ================================
# Third-party
# ================================
import numpy as np
import joblib

# ================================
# Configuración global
# ================================

# ---- Warnings (controlado, no agresivo)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- Logging limpio para notebooks
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.propagate = False  # evita duplicación con el root logger

# limpiar handlers si se re-ejecuta la celda
if logger.handlers:
    logger.handlers.clear()

handler = logging.StreamHandler()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

logger.info("Environment initialized")

2026-04-23 14:50:42,570 | INFO | Environment initialized


## **2. Acceso a drive**

In [2]:
# ================================
# Entorno (Google Drive / local)
# ================================

from pathlib import Path
import os

# Detectar si estamos en Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    DEFAULT_DRIVE_DIR = "/content/drive/MyDrive/neural_profit/"
else:
    # fallback local (puedes ajustarlo si quieres)
    DEFAULT_DRIVE_DIR = "./neural_profit/"

# Ruta base del proyecto
DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", DEFAULT_DRIVE_DIR))

# Validación básica
if not DRIVE_DIR.exists():
    logger.warning(f"DRIVE_DIR no existe: {DRIVE_DIR}")
else:
    logger.info(f"DRIVE_DIR: {DRIVE_DIR}")

2026-04-23 14:51:03,289 | INFO | DRIVE_DIR: /content/drive/MyDrive/neural_profit


Mounted at /content/drive


## **3. Rutas de ventanas `seq2one` y `scaler`**

In [3]:
# ================================
# Configuración de paths
# ================================

WINDOWS_SEQ2ONE_DIR = DRIVE_DIR / os.environ.get(
    "WINDOWS_SEQ2ONE_DIR",
    "data/07_windows/seq2one/"
)

SCALERS_DIR = DRIVE_DIR / os.environ.get(
    "SCALERS_DIR",
    "data/06_scaled/"
)

# Validación básica
for p in [WINDOWS_SEQ2ONE_DIR, SCALERS_DIR]:
    if not p.exists():
        logger.warning(f"Path no existe: {p}")
    else:
        logger.info(f"Path OK: {p}")

# ================================
# Configuración del experimento
# ================================

# Targets T2 (clasificación)
TARGETS = [
  "t2_p40_h30",
  "t2_p40_h60",
  "t2_p50_h30",
]

# Tamaños de ventana
WINDOW_SIZES = [30]

# Splits
SPLITS = ["train", "valid", "test"]

# Features finales (consistencia global)
FEATURES_T2 = [
        "ema_60",
        "roc_60",
        "roc_30",
        "stoch_k_30",
        "mom_5",
        "atr_norm_10",
        "macd"
]

logger.info("Configuración de experimento cargada")
logger.info(f"Targets: {TARGETS}")
logger.info(f"Window sizes: {WINDOW_SIZES}")

2026-04-23 14:51:03,892 | INFO | Path OK: /content/drive/MyDrive/neural_profit/data/07_windows/seq2one
2026-04-23 14:51:03,894 | INFO | Path OK: /content/drive/MyDrive/neural_profit/data/06_scaled
2026-04-23 14:51:03,895 | INFO | Configuración de experimento cargada
2026-04-23 14:51:03,895 | INFO | Targets: ['t2_p40_h30', 't2_p40_h60', 't2_p50_h30']
2026-04-23 14:51:03,896 | INFO | Window sizes: [30]


In [4]:
# ================================
# Construcción y validación de paths (windows + scaler compartido)
# ================================

def build_windows_paths(window_sizes, targets, splits, base_dir):
    """
    windows_paths[w][t][s] -> Path
    """
    windows_paths = {}

    for w in window_sizes:
        windows_paths[w] = {}
        for t in targets:
            windows_paths[w][t] = {}
            for s in splits:
                windows_paths[w][t][s] = (
                    base_dir
                    / f"L{w}"
                    / f"windows_{t}_{s}.npz"
                )

    return windows_paths


def build_shared_scaler_path(base_dir, scaler_name="scaler_t2.pkl"):
    """
    Retorna el path del scaler compartido para T2.
    """
    return base_dir / scaler_name


# ================================
# Construcción
# ================================

WINDOWS_PATHS = build_windows_paths(
    window_sizes=WINDOW_SIZES,
    targets=TARGETS,
    splits=SPLITS,
    base_dir=WINDOWS_SEQ2ONE_DIR,
)

SCALER_T2_PATH = build_shared_scaler_path(
    base_dir=SCALERS_DIR,
    scaler_name="scaler_mnq_t2.pkl",
)

logger.info("Paths construidos (windows + scaler compartido T2)")

# ================================
# Validación
# ================================

missing_windows = []
existing_windows = []

for w in WINDOW_SIZES:
    for t in TARGETS:
        for s in SPLITS:
            p = WINDOWS_PATHS[w][t][s]
            if p.exists():
                existing_windows.append(p)
            else:
                missing_windows.append(p)

scaler_exists = SCALER_T2_PATH.exists()

# -------------------------------
# Logs
# -------------------------------
logger.info(f"Windows OK      : {len(existing_windows)}")
logger.info(f"Windows missing : {len(missing_windows)}")

if scaler_exists:
    logger.info(f"Scaler T2 OK    : {SCALER_T2_PATH}")
else:
    logger.warning(f"Scaler T2 missing: {SCALER_T2_PATH}")

if missing_windows:
    logger.warning("Archivos de ventanas faltantes:")
    for p in missing_windows[:10]:
        logger.warning(f"Missing window: {p}")

if not missing_windows and scaler_exists:
    logger.info("Todos los archivos existen correctamente")

# ================================
# Ejemplo de uso
# ================================

# window
# WINDOWS_PATHS[30]["t2_p40_h30"]["train"]

# scaler compartido
# SCALER_T2_PATH

2026-04-23 14:51:03,906 | INFO | Paths construidos (windows + scaler compartido T2)
2026-04-23 14:51:04,543 | INFO | Windows OK      : 9
2026-04-23 14:51:04,544 | INFO | Windows missing : 0
2026-04-23 14:51:04,544 | INFO | Scaler T2 OK    : /content/drive/MyDrive/neural_profit/data/06_scaled/scaler_mnq_t2.pkl
2026-04-23 14:51:04,545 | INFO | Todos los archivos existen correctamente


## **4. Carga de ventanas X/y**

### **4.1. Cargar ventanas (*.npz)**

In [5]:
from pathlib import Path
from typing import Tuple
import numpy as np


def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar (T2 seq2one).

    Espera:
    - 'X': (n_samples, seq_len, n_features)
    - 'y' o 'Y': (n_samples,) o (n_samples, 1) o (n_samples, seq_len, 1)

    Retorna
    -------
    X : np.ndarray  -> (n_samples, seq_len, n_features)
    y : np.ndarray  -> (n_samples,)
    """

    # --------------------------------------------------
    # 1. Validación
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")

        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # copiar a memoria
        X = X.copy()
        y = y.copy()

    # --------------------------------------------------
    # 3. Ajuste de shape de y (CRÍTICO)
    # --------------------------------------------------
    # Caso (n_samples, 1)
    if y.ndim == 2 and y.shape[1] == 1:
        y = y.reshape(-1)

    # Caso (n_samples, seq_len, 1) -> tomar último valor
    elif y.ndim == 3:
        y = y[:, -1, 0]

    # Caso (n_samples, seq_len) -> tomar último valor
    elif y.ndim == 2 and y.shape[1] > 1:
        y = y[:, -1]

    # Validación final
    if y.ndim != 1:
        raise ValueError(f"y no es 1D después de procesamiento: shape={y.shape}")

    # --------------------------------------------------
    # 4. Logs útiles
    # --------------------------------------------------
    logger.info(f"Loaded: {path.name}")
    logger.info(f"X shape: {X.shape} | y shape: {y.shape}")

    return X, y

### **4.2. Cargar escalador (*.pkl)**

In [6]:
# --------------------------------------------------
# Función común: carga scaler compartido (T2)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib.

    Parámetros
    ----------
    path : Path
        Ruta al archivo scaler (.pkl)

    Retorna
    -------
    scaler : Any
        Objeto scaler (ej. StandardScaler, MinMaxScaler)
    """

    # --------------------------------------------------
    # 1. Validación de existencia
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    scaler = joblib.load(path)

    # --------------------------------------------------
    # 3. Validación básica (opcional pero útil)
    # --------------------------------------------------
    if not hasattr(scaler, "transform"):
        raise TypeError(f"El objeto cargado no es un scaler válido: {type(scaler)}")

    # --------------------------------------------------
    # 4. Logging
    # --------------------------------------------------
    logger.info(f"Scaler cargado: {path.name}")

    return scaler

### **4.3. Función de carga**

In [7]:
from typing import Any, Dict, Mapping
from pathlib import Path


# --------------------------------------------------
# Carga completa: ventanas + scaler compartido T2
# --------------------------------------------------
def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scaler_path: Path,
) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler compartido T2
    para un par (window_size, target).

    Parámetros
    ----------
    window_size : int
        Tamaño de ventana.
    target : str
        Target T2, por ejemplo:
        - 't2_p40_h30'
        - 't2_p40_h60'
        - 't2_p50_h30'
    windows_paths : mapping
        Estructura:
        windows_paths[window_size][target][split] -> Path
    scaler_path : Path
        Ruta al scaler compartido T2.

    Retorna
    -------
    bundle : dict
        Diccionario con:
        - metadata
        - paths
        - scaler
        - train/valid/test con X e y
    """

    # --------------------------
    # 1) Validaciones
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    for split in ["train", "valid", "test"]:
        if split not in windows_paths[window_size][target]:
            raise KeyError(
                f"split='{split}' no existe en windows_paths[{window_size}]['{target}']"
            )

    if not scaler_path.exists():
        raise FileNotFoundError(f"No se encontró el scaler compartido: {scaler_path}")

    # --------------------------
    # 2) Paths
    # --------------------------
    train_path = windows_paths[window_size][target]["train"]
    valid_path = windows_paths[window_size][target]["valid"]
    test_path  = windows_paths[window_size][target]["test"]

    # --------------------------
    # 3) Carga de ventanas
    # --------------------------
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test, y_test   = load_npz_windows(test_path)

    # --------------------------
    # 4) Carga de scaler
    # --------------------------
    scaler = load_scaler(scaler_path)

    # --------------------------
    # 5) Inferir horizonte
    # --------------------------
    try:
        horizon_str = target.split("_")[-1]   # ej: "h30"
        horizon = int(horizon_str.replace("h", ""))
    except Exception:
        horizon = None

    # --------------------------
    # 6) Logging
    # --------------------------
    logger.info(
        f"Bundle cargado | target={target} | window_size={window_size} | "
        f"train={X_train.shape} | valid={X_valid.shape} | test={X_test.shape}"
    )

    # --------------------------
    # 7) Retorno
    # --------------------------
    return {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {
            "X": X_train,
            "y": y_train,
        },
        "valid": {
            "X": X_valid,
            "y": y_valid,
        },
        "test": {
            "X": X_test,
            "y": y_test,
        },
    }

### **4.4. Creación de bundles T2**

In [8]:
def create_bundles(
    window_size: int,
    targets: list[str] = TARGETS,
    windows_paths=WINDOWS_PATHS,
    scaler_path=SCALER_T2_PATH,
):
    """
    Crea un bundle por target para un window_size dado.

    Retorna
    -------
    bundles : dict
        bundles[target] -> bundle dict
    """

    bundles = {}

    # --------------------------
    # Construcción
    # --------------------------
    for target in targets:
        bundles[target] = load_windows_and_scaler(
            window_size=window_size,
            target=target,
            windows_paths=windows_paths,
            scaler_path=scaler_path,
        )

    # --------------------------
    # Verificación rápida
    # --------------------------
    print(f"\n{'=' * 70}")
    print(f"WINDOW_SIZE: {window_size}")
    print(f"{'=' * 70}")

    for target in targets:
        b = bundles[target]

        print(f"\nTARGET: {target}")
        print("Train :", b["train"]["X"].shape, b["train"]["y"].shape)
        print("Valid :", b["valid"]["X"].shape, b["valid"]["y"].shape)
        print("Test  :", b["test"]["X"].shape,  b["test"]["y"].shape)
        print("Scaler:", type(b["scaler"]).__name__)

    return bundles

In [9]:
bundles_L30 = create_bundles(window_size=30)

2026-04-23 14:51:05,092 | INFO | Loaded: windows_t2_p40_h30_train.npz
2026-04-23 14:51:05,093 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-23 14:51:05,358 | INFO | Loaded: windows_t2_p40_h30_valid.npz
2026-04-23 14:51:05,359 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-23 14:51:05,639 | INFO | Loaded: windows_t2_p40_h30_test.npz
2026-04-23 14:51:05,640 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-23 14:51:06,868 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-23 14:51:06,869 | INFO | Bundle cargado | target=t2_p40_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | test=(6913, 30, 7)
2026-04-23 14:51:07,419 | INFO | Loaded: windows_t2_p40_h60_train.npz
2026-04-23 14:51:07,420 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-23 14:51:07,801 | INFO | Loaded: windows_t2_p40_h60_valid.npz
2026-04-23 14:51:07,802 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-23 14:51:08,062 | INFO | Loaded: windows_t2_p4


WINDOW_SIZE: 30

TARGET: t2_p40_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

TARGET: t2_p40_h60
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

TARGET: t2_p50_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler


Como acceder a las ventanas X e y:

```python
bundle_p40_h30 = bundles_L30["t2_p40_h30"]

X_train = bundle_p40_h30["train"]["X"]
y_train = bundle_p40_h30["train"]["y"]

X_valid = bundle_p40_h30["valid"]["X"]
y_valid = bundle_p40_h30["valid"]["y"]

X_test = bundle_p40_h30["test"]["X"]
y_test = bundle_p40_h30["test"]["y"]

scaler = bundle_p40_h30["scaler"]

print("Target :", bundle_p40_h30["target"])
print("Horizon:", bundle_p40_h30["horizon"])
print("Train  :", X_train.shape, y_train.shape)
print("Valid  :", X_valid.shape, y_valid.shape)
print("Test   :", X_test.shape, y_test.shape)
print("Scaler :", type(scaler).__name__)
```




In [10]:
bundles_L30

{'t2_p40_h30': {'window_size': 30,
  'target': 't2_p40_h30',
  'horizon': 30,
  'paths': {'train': '/content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_p40_h30_train.npz',
   'valid': '/content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_p40_h30_valid.npz',
   'test': '/content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_p40_h30_test.npz',
   'scaler': '/content/drive/MyDrive/neural_profit/data/06_scaled/scaler_mnq_t2.pkl'},
  'scaler': StandardScaler(),
  'train': {'X': array([[[-0.11680385, -0.06583842, -0.2238865 , ..., -0.04129868,
            -1.507039  , -0.02778307],
           [ 0.0909589 ,  0.05818945, -0.07248875, ...,  0.3872164 ,
            -1.3723946 ,  0.08631181],
           [-0.18229802, -0.13053213, -0.21077001, ..., -0.1151728 ,
            -1.2194692 ,  0.02307655],
           ...,
           [ 0.40151003,  0.28442496,  0.59289074, ...,  0.47497907,
            -1.0690751 , -0.19294474],
         

### **4.5. Preparación de inputs según el tipo de modelo**

In [11]:
# ============================================================
# Preparación de inputs según el tipo de modelo
# ============================================================

def flatten_seq2one_X(X: np.ndarray) -> np.ndarray:
    """
    Aplana ventanas seq2one para modelos tabulares.

    Convierte:
        (n_samples, seq_len, n_features)
    a:
        (n_samples, seq_len * n_features)
    """
    X = np.asarray(X)

    if X.ndim != 3:
        raise ValueError(
            f"Se esperaba X 3D con shape (n, seq_len, n_features). "
            f"Recibido X.shape={X.shape}"
        )

    n_samples = X.shape[0]
    return X.reshape(n_samples, -1)


def prepare_X_for_model(
    X: np.ndarray,
    *,
    input_mode: str,
) -> np.ndarray:
    """
    Prepara X según el tipo de modelo.

    Parámetros
    ----------
    X : np.ndarray
        Array de entrada.
    input_mode : str
        - '2d_flat' : aplana ventanas 3D a 2D
        - '3d'      : deja X como está

    Retorna
    -------
    np.ndarray
        X transformado para el modelo correspondiente.
    """
    X = np.asarray(X)

    if input_mode == "3d":
        if X.ndim != 3:
            raise ValueError(
                f"Se esperaba X 3D para input_mode='3d'. "
                f"Recibido X.shape={X.shape}"
            )
        return X

    if input_mode == "2d_flat":
        return flatten_seq2one_X(X)

    raise ValueError(
        f"input_mode no soportado: {input_mode}. "
        f"Use '2d_flat' o '3d'."
    )

**Cómo se usa después**

Para Logistic Regression:

```python
X_train_model = prepare_X_for_model(bundle["train"]["X"], input_mode="2d_flat")
X_valid_model = prepare_X_for_model(bundle["valid"]["X"], input_mode="2d_flat")
```

Para LSTM / GRU / Transformer:

```python
X_train_model = prepare_X_for_model(bundle["train"]["X"], input_mode="3d")
X_valid_model = prepare_X_for_model(bundle["valid"]["X"], input_mode="3d")
```


## **5. Sanity Check**

Estos sanity checks sirven para verificar, antes de entrenar, que los datos cargados tengan la estructura correcta y no vengan con errores silenciosos.

En concreto, comprueban que:

- X tenga el formato esperado: 3D (n, seq_len, n_features) o 2D (n, d_flat)
- y tenga forma válida para clasificación seq2one
- X e y tengan la misma cantidad de muestras
- no haya NaN ni inf
- las dimensiones sean consistentes entre train, valid y test
- exista más de una clase en y

Nos conviene tenerlos, porque ayudan a detectar errores de shape o de datos antes de llegar al entrenamiento.

In [12]:
from __future__ import annotations

from typing import Any, Optional, Tuple, Dict, Mapping
import numpy as np


# ============================================================
# SANITY CHECKS PARA DATASETS SEQ2ONE (T2)
# ============================================================

def _as_numpy_array(a: Any, *, name: str) -> np.ndarray:
    """
    Convierte una entrada a np.ndarray y valida que no esté vacía
    ni contenga NaN/inf.
    """
    arr = np.asarray(a)

    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")

    if not np.isfinite(arr).all():
        raise ValueError(
            f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}"
        )

    return arr


def _normalize_y_seq2one(
    y: np.ndarray,
    *,
    name: str = "y",
    allow_seq_inputs_take_last: bool = False,
) -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).

    Acepta:
    - (n,)
    - (n, 1)
    - (n, seq_len)       si allow_seq_inputs_take_last=True
    - (n, seq_len, 1)    si allow_seq_inputs_take_last=True
    """
    if y.ndim == 1:
        return y

    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)

    if allow_seq_inputs_take_last:
        if y.ndim == 2 and y.shape[1] > 1:
            return y[:, -1]

        if y.ndim == 3 and y.shape[2] == 1:
            return y[:, -1, 0]

    raise ValueError(
        f"{name} shape inválido para seq2one. "
        f"Se esperaba (n,) o (n,1)"
        f"{' o secuencial si allow_seq_inputs_take_last=True' if allow_seq_inputs_take_last else ''}. "
        f"Recibido {y.shape}"
    )


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiere (seq_len, n_features, mode) desde X.

    mode:
    - '3d': X = (n, seq_len, n_features)
    - '2d': X = (n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"

    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"

    raise ValueError(
        f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}"
    )


def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one de clasificación.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)
      - y secuencial, opcionalmente, tomando el último valor

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: no aplica directamente.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D.
    allow_seq_inputs_take_last:
        Si y viene como secuencia, toma el último valor.
    """
    X = _as_numpy_array(X, name=f"X[{split_name}]")
    y = _as_numpy_array(y, name=f"y[{split_name}]")

    y = _normalize_y_seq2one(
        y,
        name=f"y[{split_name}]",
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
    )

    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: "
            f"X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    if mode == "3d":
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, "
                f"recibido={seq_len}. X.shape={X.shape}"
            )

        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, "
                f"recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, "
                f"recibido={d_flat}. X.shape={X.shape}"
            )

        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim o pase X en 3D."
            )

    classes, counts = np.unique(y, return_counts=True)
    class_distribution = {
        str(cls): int(cnt) for cls, cnt in zip(classes, counts)
    }

    if len(classes) < 2:
        raise ValueError(
            f"{split_name}: y contiene menos de 2 clases únicas. "
            f"classes={classes.tolist()}"
        )

    info = {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "n_classes": int(len(classes)),
        "classes": classes.tolist(),
        "class_distribution": class_distribution,
    }

    if verbose:
        print(
            f"[sanity_check_seq2one] {split_name} | "
            f"X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | n_classes={info['n_classes']} | "
            f"classes={info['classes']}"
        )

    return info


def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "target": "t2_p40_h30",
      "horizon": 30,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Si no se pasan expected_*, usa TRAIN como referencia.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        expected_n_features = None

    out_tr = sanity_check_seq2one(
        X_tr,
        y_tr,
        f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
        verbose=verbose,
    )

    out_va = sanity_check_seq2one(
        X_va,
        y_va,
        f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
        verbose=verbose,
    )

    out_te = sanity_check_seq2one(
        X_te,
        y_te,
        f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
        verbose=verbose,
    )

    target = bundle.get("target", "NA")
    horizon = bundle.get("horizon", "NA")

    if verbose:
        print(f"OK {tag} | target={target} | horizon={horizon}")

    return {
        "target": target,
        "horizon": horizon,
        "train": out_tr,
        "valid": out_va,
        "test": out_te,
    }


def run_sanity_checks_all_bundles_seq2one(
    bundles: Mapping[str, Dict[str, Any]],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Corre sanity checks para todos los bundles de un diccionario:

    bundles[target] -> bundle
    """
    results = {}

    for target, bundle in bundles.items():
        results[target] = run_sanity_checks_for_bundle_seq2one(
            bundle,
            tag=target,
            verbose=verbose,
        )

    return results

In [13]:
sanity_results = run_sanity_checks_all_bundles_seq2one(
    bundles_L30,
    verbose=True,
)

[sanity_check_seq2one] train_t2_p40_h30 | X=(32147, 30, 7) | y=(32147,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] valid_t2_p40_h30 | X=(6882, 30, 7) | y=(6882,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] test_t2_p40_h30 | X=(6913, 30, 7) | y=(6913,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
OK t2_p40_h30 | target=t2_p40_h30 | horizon=30
[sanity_check_seq2one] train_t2_p40_h60 | X=(32147, 30, 7) | y=(32147,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] valid_t2_p40_h60 | X=(6882, 30, 7) | y=(6882,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] test_t2_p40_h60 | X=(6913, 30, 7) | y=(6913,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
OK t2_p40_h60 | target=t2_p40_h60 | horizon=60
[sanity_check_seq2one] train_t2_p50_h30 | X=(32147, 30, 7) | y=(32147,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] valid_t2_p50_h30 | X=(6882, 30, 7) | y=(6882,) | mode=3d | n_classes=3 | c

## **6. Módulo de métricas T2**

In [14]:
# ================================
# Setup para importar módulos del proyecto
# ================================

import sys
import importlib

if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))

# asegurar que metrics es paquete
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

importlib.invalidate_caches()

# ================================
# Imports de métricas
# ================================

from metrics.classification_metrics import (
    compute_classification_metrics,
    metrics_to_flat_dict,
    print_classification_report_block,
)

from metrics.classification_probabilities import (
    compute_probabilistic_outputs,
    apply_decision_rule,
)

print("Módulos importados correctamente")

Módulos importados correctamente


In [15]:
# ================================
# Utilidades: outputs -> DataFrame
# ================================

from __future__ import annotations

import pandas as pd
from typing import Any


def classification_metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    window_size: int,
    target: str,
    horizon: int | None = None,
) -> pd.DataFrame:
    """
    Convierte el output de compute_classification_metrics(...)
    en una fila de DataFrame.

    Usa metrics_to_flat_dict(...) para aplanar la salida
    del módulo classification_metrics y luego agrega metadata
    del experimento.
    """
    flat_metrics = metrics_to_flat_dict(metrics)

    row = {
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        **flat_metrics,
    }

    return pd.DataFrame([row])


def _flatten_dict(
    d: dict[str, Any],
    *,
    parent_key: str = "",
    sep: str = "_",
) -> dict[str, Any]:
    """
    Aplana un diccionario arbitrario de forma recursiva.

    Ejemplo:
    {"a": {"b": 1}} -> {"a_b": 1}
    """
    items: dict[str, Any] = {}

    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else str(k)

        if isinstance(v, dict):
            items.update(_flatten_dict(v, parent_key=new_key, sep=sep))
        else:
            items[new_key] = v

    return items


def probabilities_metrics_to_df(
    prob_metrics: dict,
    *,
    model: str,
    split: str,
    window_size: int,
    target: str,
    horizon: int | None = None,
) -> pd.DataFrame:
    """
    Convierte el output del módulo classification_probabilities
    en una fila de DataFrame.

    Como la estructura puede variar según la implementación,
    se aplana recursivamente el diccionario y luego se agrega
    metadata del experimento.
    """
    flat_prob_metrics = _flatten_dict(prob_metrics)

    row = {
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        **flat_prob_metrics,
    }

    return pd.DataFrame([row])


logger.info("Utilidades de exportación a DataFrame cargadas")

2026-04-23 14:51:11,455 | INFO | Utilidades de exportación a DataFrame cargadas


## **7. Persistencia de métricas (tracking de experimentos)**

In [16]:
# ================================
# Persistencia de métricas de clasificación
# ================================

from pathlib import Path
import pandas as pd


def load_classification_metrics_if_exists(
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics_tuning" / "classification_metrics",
) -> pd.DataFrame:
    """
    Carga métricas de clasificación si el archivo existe.

    El nombre del archivo incluye modelo y split para evitar
    mezclar resultados de distintos experimentos.
    """
    path = base_dir / f"classification_metrics_{model_name}_{split}.parquet"

    if path.exists():
        logger.info(f"Cargando métricas desde: {path}")
        return pd.read_parquet(path)

    logger.info(
        f"No existen métricas previas para model={model_name} | split={split}"
    )
    return pd.DataFrame()


def save_classification_metrics(
    df_metrics: pd.DataFrame,
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics_tuning" / "classification_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas de clasificación en formato Parquet.
    """
    base_dir.mkdir(parents=True, exist_ok=True)

    out_path = base_dir / f"classification_metrics_{model_name}_{split}.parquet"
    df_metrics.to_parquet(out_path, index=False)

    logger.info(f"Métricas guardadas en: {out_path}")

    return out_path


# ================================
# Persistencia de probabilidades / decisión
# ================================

def load_classification_probabilities_if_exists(
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics_tuning" / "classification_probabilities",
) -> pd.DataFrame:
    """
    Carga resultados probabilísticos si el archivo existe.

    El nombre del archivo incluye modelo y split para evitar
    mezclar resultados de distintos experimentos.
    """
    path = base_dir / f"classification_probabilities_{model_name}_{split}.parquet"

    if path.exists():
        logger.info(f"Cargando probabilidades desde: {path}")
        return pd.read_parquet(path)

    logger.info(
        f"No existen probabilidades previas para model={model_name} | split={split}"
    )
    return pd.DataFrame()


def save_classification_probabilities(
    df_probabilities: pd.DataFrame,
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics_tuning" / "classification_probabilities",
) -> Path:
    """
    Guarda un DataFrame de probabilidades / decisión en formato Parquet.
    """
    base_dir.mkdir(parents=True, exist_ok=True)

    out_path = base_dir / f"classification_probabilities_{model_name}_{split}.parquet"
    df_probabilities.to_parquet(out_path, index=False)

    logger.info(f"Probabilidades guardadas en: {out_path}")

    return out_path

Ejemplo de uso con logistic regression:

```python
df_metrics_all = load_classification_metrics_if_exists(
    model_name="logistic_regression",
    split="valid",
)

df_probabilities_all = load_classification_probabilities_if_exists(
    model_name="logistic_regression",
    split="valid",
)
```

Guardar:
```python
save_classification_metrics(
    df_metrics_all,
    model_name="logistic_regression",
    split="valid",
)

save_classification_probabilities(
    df_probabilities_all,
    model_name="logistic_regression",
    split="valid",
)
```

## **8. Gestión de dispositivo y memoria**

In [17]:
# ================================
# Device y limpieza de memoria
# ================================

def get_torch_device():
    """
    Retorna el device para modelos PyTorch.
    """
    import torch

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Using device: {device}")
    return device


def clear_torch_memory() -> None:
    """
    Limpia memoria Python y, si existe CUDA, libera caché GPU.
    Útil entre entrenamientos de modelos PyTorch.
    """
    import gc

    gc.collect()

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass

## **9. Reproducibilidad**

In [18]:
# ================================
# Reproducibilidad
# ================================

def set_seeds(seed: int = 42) -> None:
    """
    Fija semillas para reproducibilidad en:
    - Python
    - NumPy
    - PyTorch (si está disponible)
    """

    import random
    import numpy as np
    import os

    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    # ---- PyTorch (si está instalado)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        # determinismo (más lento pero reproducible)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    except ImportError:
        pass  # no hay PyTorch, ignorar

    logger.info(f"Seeds fijadas en {seed}")


# Aplicación
SEED = 42
set_seeds(SEED)

2026-04-23 14:51:18,216 | INFO | Seeds fijadas en 42


# **10. Entrenamiento de modelo XGBoost**

## **10.1. Función unitaria por bundle**

In [19]:
from xgboost import XGBClassifier
import numpy as np


def run_xgboost_for_bundle_seq2one(
    bundle,
    *,
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=1.0,
    colsample_bytree=1.0,
    min_child_weight=1,
    gamma=0.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    input_mode="2d_flat",
    class_weight=None,
    tree_method="hist",
    device="cuda",
    verbose=False,
):

    # =========================
    # 1. DATA
    # =========================
    X_train = bundle["train"]["X"]
    y_train = bundle["train"]["y"]

    X_valid = bundle["valid"]["X"]
    y_valid = bundle["valid"]["y"]

    target = bundle.get("target")
    horizon = bundle.get("horizon")
    window_size = bundle.get("window_size")

    # =========================
    # 2. INPUT
    # =========================
    X_train_model = prepare_X_for_model(X_train, input_mode=input_mode)
    X_valid_model = prepare_X_for_model(X_valid, input_mode=input_mode)

    # =========================
    # 3. LABEL ENCODING
    # =========================
    classes_ = np.sort(np.unique(y_train))
    class_to_idx = {cls: idx for idx, cls in enumerate(classes_)}
    idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

    y_train_enc = np.array([class_to_idx[y] for y in y_train], dtype=np.int32)
    num_class = len(classes_)

    # Validación opcional
    if not np.array_equal(classes_, [-1, 0, 1]):
        raise ValueError(f"Clases inesperadas: {classes_}")

    # =========================
    # 4. SAMPLE WEIGHTS
    # =========================
    sample_weight = None

    if class_weight == "balanced":
        counts = np.bincount(y_train_enc, minlength=num_class)
        total = counts.sum()

        weights = total / (num_class * counts)
        sample_weight = weights[y_train_enc]

    elif isinstance(class_weight, dict):
        weights = {class_to_idx[k]: v for k, v in class_weight.items()}
        sample_weight = np.array([weights.get(i, 1.0) for i in y_train_enc])

    elif class_weight is not None:
        raise ValueError("class_weight debe ser None, 'balanced' o dict")

    # =========================
    # 5. MODEL
    # =========================
    model = XGBClassifier(
        objective="multi:softprob",
        num_class=num_class,
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_child_weight=min_child_weight,
        gamma=gamma,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        random_state=random_state,
        n_jobs=n_jobs,
        eval_metric="mlogloss",
        tree_method=tree_method,
        device=device,
        verbosity=0,
    )

    # =========================
    # 6. TRAIN
    # =========================
    model.fit(
        X_train_model,
        y_train_enc,
        sample_weight=sample_weight,
    )

    # =========================
    # 7. PREDICT
    # =========================
    y_pred_valid_enc = model.predict(X_valid_model)
    y_pred_valid = np.array([idx_to_class[int(y)] for y in y_pred_valid_enc])

    y_proba_valid = model.predict_proba(X_valid_model)

    # =========================
    # 8. RETURN
    # =========================
    return {
        "model_name": "xgboost",
        "target": target,
        "horizon": horizon,
        "window_size": window_size,
        "input_mode": input_mode,
        "class_weight": str(class_weight),
        "tree_method": tree_method,
        "device": device,

        # hiperparámetros
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "learning_rate": learning_rate,
        "subsample": subsample,
        "colsample_bytree": colsample_bytree,
        "min_child_weight": min_child_weight,
        "gamma": gamma,
        "reg_alpha": reg_alpha,
        "reg_lambda": reg_lambda,
        "random_state": random_state,
        "n_jobs": n_jobs,

        # modelo y outputs
        "model": model,
        "classes_": classes_.tolist(),
        "y_valid": y_valid,
        "y_pred_valid": y_pred_valid,
        "y_proba_valid": y_proba_valid,
    }

## **10.2. Función de evaluación sobre uno o más bundles**

In [20]:
from typing import Any, Dict, List, Sequence, Union
import pandas as pd


def eval_xgboost_bundles(
    bundles: Union[Dict[str, Any], Sequence[Dict[str, Any]]],
    *,
    model_name: str = "xgboost",
    n_estimators: int = 100,
    max_depth: int = 6,
    learning_rate: float = 0.1,
    subsample: float = 1.0,
    colsample_bytree: float = 1.0,
    min_child_weight: float = 1.0,
    gamma: float = 0.0,
    reg_alpha: float = 0.0,
    reg_lambda: float = 1.0,
    random_state: int = 42,
    n_jobs: int = -1,
    input_mode: str = "2d_flat",
    class_weight=None,
    tree_method: str = "hist",
    device: str = "cuda",
    prob_threshold_long: float = 0.40,
    prob_threshold_short: float = 0.40,
    verbose: bool = False,
) -> Dict[str, pd.DataFrame]:

    # --------------------------------------------------
    # 1) Normalizar entrada
    # --------------------------------------------------
    if isinstance(bundles, dict):
        if "train" in bundles and "valid" in bundles:
            bundles_list: List[Dict[str, Any]] = [bundles]
        else:
            bundles_list = list(bundles.values())
    else:
        bundles_list = list(bundles)

    metrics_rows = []
    probabilities_rows = []

    # --------------------------------------------------
    # 2) Iterar por bundles
    # --------------------------------------------------
    for bundle in bundles_list:
        target = bundle.get("target", None)
        window_size = int(bundle["window_size"])
        horizon = int(bundle.get("horizon", -1))

        if verbose:
            print(
                f"  -> L{window_size} | "
                f"target={target} | "
                f"model={model_name} | "
                f"class_weight={class_weight}"
            )

        # --------------------------------------------------
        # 3) Entrenar modelo
        # --------------------------------------------------
        preds = run_xgboost_for_bundle_seq2one(
            bundle,
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            min_child_weight=min_child_weight,
            gamma=gamma,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            random_state=random_state,
            n_jobs=n_jobs,
            input_mode=input_mode,
            class_weight=class_weight,
            tree_method=tree_method,
            device=device,
            verbose=False,
        )

        y_true = preds["y_valid"]
        y_pred = preds["y_pred_valid"]
        y_proba = preds["y_proba_valid"]

        # --------------------------------------------------
        # 4) Métricas de clasificación
        # --------------------------------------------------
        metrics = compute_classification_metrics(
            y_true=y_true,
            y_pred=y_pred,
            model_name=model_name,
            split="valid",
            target=target,
            labels=[-1, 0, 1],
        )

        df_metrics_row = classification_metrics_to_df(
            metrics,
            model=model_name,
            split="valid",
            window_size=window_size,
            target=target,
            horizon=horizon,
        )

        # class_weight mode
        if class_weight == "balanced":
            class_weight_mode = "balanced"
        elif class_weight is None:
            class_weight_mode = "none"
        else:
            class_weight_mode = "custom"

        # metadata completa
        df_metrics_row["class_weight_mode"] = class_weight_mode
        df_metrics_row["input_mode"] = input_mode
        df_metrics_row["n_estimators"] = n_estimators
        df_metrics_row["max_depth"] = max_depth
        df_metrics_row["learning_rate"] = learning_rate
        df_metrics_row["subsample"] = subsample
        df_metrics_row["colsample_bytree"] = colsample_bytree
        df_metrics_row["min_child_weight"] = min_child_weight
        df_metrics_row["gamma"] = gamma
        df_metrics_row["reg_alpha"] = reg_alpha
        df_metrics_row["reg_lambda"] = reg_lambda
        df_metrics_row["tree_method"] = tree_method
        df_metrics_row["device"] = device
        df_metrics_row["n_jobs"] = n_jobs
        df_metrics_row["random_state"] = random_state
        df_metrics_row["threshold_long"] = prob_threshold_long
        df_metrics_row["threshold_short"] = prob_threshold_short

        metrics_rows.append(df_metrics_row)

        # --------------------------------------------------
        # 5) Outputs probabilísticos (FIX CRÍTICO)
        # --------------------------------------------------
        class_labels = preds["classes_"]

        proba_df = compute_probabilistic_outputs(
            y_proba=y_proba,
            class_labels=class_labels,
            y_true=y_true,
        )

        decision_df = apply_decision_rule(
            proba_df,
            long_class=1,
            short_class=-1,
            long_threshold=prob_threshold_long,
            short_threshold=prob_threshold_short,
        )

        overlap_cols = [c for c in decision_df.columns if c in proba_df.columns]
        if overlap_cols:
            decision_df = decision_df.drop(columns=overlap_cols)

        df_prob = pd.concat(
            [proba_df.reset_index(drop=True), decision_df.reset_index(drop=True)],
            axis=1,
        )

        # --------------------------------------------------
        # 6) Metadata + sample_id (CRÍTICO)
        # --------------------------------------------------
        df_prob["model"] = model_name
        df_prob["split"] = "valid"
        df_prob["window_size"] = window_size
        df_prob["target"] = target
        df_prob["horizon"] = horizon
        df_prob["class_weight_mode"] = class_weight_mode
        df_prob["input_mode"] = input_mode
        df_prob["n_estimators"] = n_estimators
        df_prob["max_depth"] = max_depth
        df_prob["learning_rate"] = learning_rate
        df_prob["subsample"] = subsample
        df_prob["colsample_bytree"] = colsample_bytree
        df_prob["min_child_weight"] = min_child_weight
        df_prob["gamma"] = gamma
        df_prob["reg_alpha"] = reg_alpha
        df_prob["reg_lambda"] = reg_lambda
        df_prob["tree_method"] = tree_method
        df_prob["device"] = device
        df_prob["n_jobs"] = n_jobs
        df_prob["random_state"] = random_state
        df_prob["threshold_long"] = prob_threshold_long
        df_prob["threshold_short"] = prob_threshold_short

        # sample_id para incremental
        df_prob = df_prob.reset_index(drop=True)
        df_prob["sample_id"] = df_prob.index.astype(int)

        probabilities_rows.append(df_prob)

    # --------------------------------------------------
    # 7) Consolidar salida
    # --------------------------------------------------
    df_metrics_all = pd.concat(metrics_rows, ignore_index=True)
    df_probabilities_all = pd.concat(probabilities_rows, ignore_index=True)

    return {
        "metrics": df_metrics_all,
        "probabilities": df_probabilities_all,
    }

## **10.3. Función orquestadora**

In [21]:
import gc
import pandas as pd


def run_xgboost(
    window_size: int,
    *,
    targets: list[str] = TARGETS,
    verbose: bool = True,
    model_name: str = "xgboost",
    n_estimators: int = 100,
    max_depth: int = 6,
    learning_rate: float = 0.1,
    subsample: float = 1.0,
    colsample_bytree: float = 1.0,
    min_child_weight: float = 1.0,
    gamma: float = 0.0,
    reg_alpha: float = 0.0,
    reg_lambda: float = 1.0,
    random_state: int = 42,
    n_jobs: int = -1,
    input_mode: str = "2d_flat",
    class_weight=None,
    tree_method: str = "hist",
    device: str = "cuda",
    prob_threshold_long: float = 0.40,
    prob_threshold_short: float = 0.40,
) -> dict[str, pd.DataFrame]:

    size = int(window_size)

    bundles = None
    results = None

    # ✔ FIX: no modificar nombre del modelo
    model_name_effective = model_name

    try:
        # --------------------------------------------------
        # 1) Header
        # --------------------------------------------------
        if verbose:
            print("\n" + "=" * 80)
            print(f"XGBOOST | T2 SEQ2ONE | WINDOW_SIZE=L{size}")
            print("=" * 80)
            print(f"targets            = {targets}")
            print(f"class_weight       = {class_weight}")
            print(f"input_mode         = {input_mode}")
            print(f"n_estimators       = {n_estimators}")
            print(f"max_depth          = {max_depth}")
            print(f"learning_rate      = {learning_rate}")
            print(f"subsample          = {subsample}")
            print(f"colsample_bytree   = {colsample_bytree}")
            print(f"min_child_weight   = {min_child_weight}")
            print(f"gamma              = {gamma}")
            print(f"reg_alpha          = {reg_alpha}")
            print(f"reg_lambda         = {reg_lambda}")
            print(f"tree_method        = {tree_method}")
            print(f"device             = {device}")
            print(f"n_jobs             = {n_jobs}")
            print(f"thr_long           = {prob_threshold_long}")
            print(f"thr_short          = {prob_threshold_short}")

        # --------------------------------------------------
        # 2) Bundles
        # --------------------------------------------------
        if verbose:
            print(f"\n[BUILD] L{size} | n_targets={len(targets)}")

        bundles = create_bundles(
            window_size=size,
            targets=targets,
            windows_paths=WINDOWS_PATHS,
            scaler_path=SCALER_T2_PATH,
        )

        # --------------------------------------------------
        # 3) Evaluación
        # --------------------------------------------------
        if verbose:
            print(
                f"\n[EVAL] L{size} | split=valid | "
                f"model={model_name_effective} | class_weight={class_weight}"
            )

        results = eval_xgboost_bundles(
            bundles=bundles,
            model_name=model_name_effective,
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            min_child_weight=min_child_weight,
            gamma=gamma,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            random_state=random_state,
            n_jobs=n_jobs,
            input_mode=input_mode,
            class_weight=class_weight,
            tree_method=tree_method,
            device=device,
            prob_threshold_long=prob_threshold_long,
            prob_threshold_short=prob_threshold_short,
            verbose=verbose,
        )

        df_metrics = (
            results["metrics"]
            .sort_values(["window_size", "target", "split", "horizon", "model"])
            .reset_index(drop=True)
        )

        df_probabilities = (
            results["probabilities"]
            .sort_values(["window_size", "target", "split", "horizon", "model"])
            .reset_index(drop=True)
        )

        # --------------------------------------------------
        # 4) Resumen
        # --------------------------------------------------
        if verbose:
            print(
                f"\n[DONE] L{size} | "
                f"metrics_rows={len(df_metrics)} | "
                f"probabilities_rows={len(df_probabilities)}"
            )

        return {
            "metrics": df_metrics,
            "probabilities": df_probabilities,
        }

    finally:
        del bundles, results
        gc.collect()

## **10.4. Función incremental de tuneo**

In [22]:
from itertools import product
import pandas as pd


def run_xgboost_grid_incremental(
    window_size: int,
    *,
    targets: list[str],
    n_estimators_values: list[int],
    max_depth_values: list[int],
    learning_rate_values: list[float],
    subsample_values: list[float],
    colsample_bytree_values: list[float],
    min_child_weight_values: list[float],
    gamma_values: list[float],
    reg_alpha_values: list[float],
    reg_lambda_values: list[float],
    model_name: str = "xgboost",
    random_state: int = 42,
    n_jobs: int = -1,
    input_mode: str = "2d_flat",
    class_weight=None,
    tree_method: str = "hist",
    device: str = "cuda",
    prob_threshold_long_values: list[float] = [0.40],
    prob_threshold_short_values: list[float] = [0.40],
    split: str = "valid",
    verbose: bool = True,
) -> dict[str, pd.DataFrame]:

    # --------------------------------------------------
    # 0) Helpers
    # --------------------------------------------------
    def safe_eq(df, col, value):
        if col in df.columns:
            return df[col] == value
        return pd.Series(False, index=df.index)

    # --------------------------------------------------
    # 1) Cargar persistencia previa
    # --------------------------------------------------
    df_metrics_existing = load_classification_metrics_if_exists(
        model_name=model_name,
        split=split,
    )

    df_prob_existing = load_classification_probabilities_if_exists(
        model_name=model_name,
        split=split,
    )

    if class_weight == "balanced":
        class_weight_mode = "balanced"
    elif class_weight is None:
        class_weight_mode = "none"
    else:
        class_weight_mode = "custom"

    # --------------------------------------------------
    # 2) Asegurar columnas requeridas
    # --------------------------------------------------
    required_metric_cols = [
        "model", "split", "window_size", "target",
        "n_estimators", "max_depth", "learning_rate",
        "subsample", "colsample_bytree", "min_child_weight",
        "gamma", "reg_alpha", "reg_lambda",
        "input_mode", "class_weight_mode",
        "tree_method", "device", "n_jobs", "random_state",
        "threshold_long", "threshold_short"
    ]

    required_prob_cols = required_metric_cols + ["sample_id"]

    for col in required_metric_cols:
        if col not in df_metrics_existing.columns:
            df_metrics_existing[col] = None

    for col in required_prob_cols:
        if col not in df_prob_existing.columns:
            df_prob_existing[col] = None

    # --------------------------------------------------
    # 3) Definir grilla
    # --------------------------------------------------
    grid = list(product(
        n_estimators_values,
        max_depth_values,
        learning_rate_values,
        subsample_values,
        colsample_bytree_values,
        min_child_weight_values,
        gamma_values,
        reg_alpha_values,
        reg_lambda_values,
        prob_threshold_long_values,
        prob_threshold_short_values,
    ))

    if verbose:
        print("\n" + "=" * 100)
        print(f"XGBOOST GRID INCREMENTAL | L={window_size}")
        print("=" * 100)
        print(f"targets                    = {targets}")
        print(f"n_combinations             = {len(grid)}")
        print(f"n_estimators_values        = {n_estimators_values}")
        print(f"max_depth_values           = {max_depth_values}")
        print(f"learning_rate_values       = {learning_rate_values}")
        print(f"subsample_values           = {subsample_values}")
        print(f"colsample_bytree_values    = {colsample_bytree_values}")
        print(f"min_child_weight_values    = {min_child_weight_values}")
        print(f"gamma_values               = {gamma_values}")
        print(f"reg_alpha_values           = {reg_alpha_values}")
        print(f"reg_lambda_values          = {reg_lambda_values}")
        print(f"threshold_long_values      = {prob_threshold_long_values}")
        print(f"threshold_short_values     = {prob_threshold_short_values}")
        print(f"class_weight_mode          = {class_weight_mode}")

    # --------------------------------------------------
    # 4) Loop principal
    # --------------------------------------------------
    for i, (
        n_estimators,
        max_depth,
        learning_rate,
        subsample,
        colsample_bytree,
        min_child_weight,
        gamma,
        reg_alpha,
        reg_lambda,
        thr_long,
        thr_short,
    ) in enumerate(grid, start=1):

        if verbose:
            print("\n" + "-" * 100)
            print(
                f"[{i}/{len(grid)}] "
                f"n_estimators={n_estimators} | "
                f"max_depth={max_depth} | "
                f"learning_rate={learning_rate} | "
                f"subsample={subsample} | "
                f"colsample_bytree={colsample_bytree} | "
                f"min_child_weight={min_child_weight} | "
                f"gamma={gamma} | "
                f"reg_alpha={reg_alpha} | "
                f"reg_lambda={reg_lambda} | "
                f"thr_long={thr_long} | "
                f"thr_short={thr_short}"
            )

        # ----------------------------------------------
        # 4.1) Verificar si el experimento ya existe
        # ----------------------------------------------
        mask = (
            safe_eq(df_metrics_existing, "model", model_name) &
            safe_eq(df_metrics_existing, "split", split) &
            safe_eq(df_metrics_existing, "window_size", window_size) &
            safe_eq(df_metrics_existing, "n_estimators", n_estimators) &
            safe_eq(df_metrics_existing, "max_depth", max_depth) &
            safe_eq(df_metrics_existing, "learning_rate", learning_rate) &
            safe_eq(df_metrics_existing, "subsample", subsample) &
            safe_eq(df_metrics_existing, "colsample_bytree", colsample_bytree) &
            safe_eq(df_metrics_existing, "min_child_weight", min_child_weight) &
            safe_eq(df_metrics_existing, "gamma", gamma) &
            safe_eq(df_metrics_existing, "reg_alpha", reg_alpha) &
            safe_eq(df_metrics_existing, "reg_lambda", reg_lambda) &
            safe_eq(df_metrics_existing, "input_mode", input_mode) &
            safe_eq(df_metrics_existing, "class_weight_mode", class_weight_mode) &
            safe_eq(df_metrics_existing, "tree_method", tree_method) &
            safe_eq(df_metrics_existing, "device", device) &
            safe_eq(df_metrics_existing, "n_jobs", n_jobs) &
            safe_eq(df_metrics_existing, "random_state", random_state) &
            safe_eq(df_metrics_existing, "threshold_long", thr_long) &
            safe_eq(df_metrics_existing, "threshold_short", thr_short)
        )

        existing_targets = set(df_metrics_existing.loc[mask, "target"].dropna().unique())
        already_exists = set(targets).issubset(existing_targets)

        if already_exists:
            if verbose:
                print("✔ Ya existe -> skip")
            continue

        # ----------------------------------------------
        # 4.2) Ejecutar modelo
        # ----------------------------------------------
        results = run_xgboost(
            window_size=window_size,
            targets=targets,
            verbose=verbose,
            model_name=model_name,
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            min_child_weight=min_child_weight,
            gamma=gamma,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            random_state=random_state,
            n_jobs=n_jobs,
            input_mode=input_mode,
            class_weight=class_weight,
            tree_method=tree_method,
            device=device,
            prob_threshold_long=thr_long,
            prob_threshold_short=thr_short,
        )

        df_metrics_new = results["metrics"].copy()
        df_prob_new = results["probabilities"].copy()

        # ----------------------------------------------
        # 4.3) Completar metadata
        # ----------------------------------------------
        df_metrics_new["random_state"] = random_state
        df_metrics_new["threshold_long"] = thr_long
        df_metrics_new["threshold_short"] = thr_short

        df_prob_new["random_state"] = random_state
        df_prob_new["threshold_long"] = thr_long
        df_prob_new["threshold_short"] = thr_short

        # sample_id por muestra dentro de cada experimento
        df_prob_new = df_prob_new.reset_index(drop=True)
        if "sample_id" not in df_prob_new.columns:
            df_prob_new["sample_id"] = df_prob_new.index.astype(int)

        # asegurar columnas requeridas
        for col in required_metric_cols:
            if col not in df_metrics_new.columns:
                df_metrics_new[col] = None

        for col in required_prob_cols:
            if col not in df_prob_new.columns:
                df_prob_new[col] = None

        # ----------------------------------------------
        # 4.4) Append
        # ----------------------------------------------
        df_metrics_existing = pd.concat(
            [df_metrics_existing, df_metrics_new],
            ignore_index=True
        )

        df_prob_existing = pd.concat(
            [df_prob_existing, df_prob_new],
            ignore_index=True
        )

        # ----------------------------------------------
        # 4.5) Deduplicación correcta
        # ----------------------------------------------
        metric_key_cols = [
            "model", "split", "window_size", "target",
            "n_estimators", "max_depth", "learning_rate",
            "subsample", "colsample_bytree", "min_child_weight",
            "gamma", "reg_alpha", "reg_lambda",
            "input_mode", "class_weight_mode",
            "tree_method", "device", "n_jobs", "random_state",
            "threshold_long", "threshold_short"
        ]

        prob_key_cols = metric_key_cols + ["sample_id"]

        df_metrics_existing = (
            df_metrics_existing
            .drop_duplicates(subset=metric_key_cols, keep="last")
            .reset_index(drop=True)
        )

        df_prob_existing = (
            df_prob_existing
            .drop_duplicates(subset=prob_key_cols, keep="last")
            .reset_index(drop=True)
        )

        # ----------------------------------------------
        # 4.6) Guardar
        # ----------------------------------------------
        save_classification_metrics(
            df_metrics_existing,
            model_name=model_name,
            split=split,
        )

        save_classification_probabilities(
            df_prob_existing,
            model_name=model_name,
            split=split,
        )

        if verbose:
            print(
                f"💾 Guardado OK | metrics={len(df_metrics_existing)} | "
                f"prob={len(df_prob_existing)}"
            )

    # --------------------------------------------------
    # 5) Retorno final
    # --------------------------------------------------
    return {
        "metrics": df_metrics_existing,
        "probabilities": df_prob_existing,
    }

# **11. Tuneo grueso**

In [23]:
results_xgb_coarse = run_xgboost_grid_incremental(
    window_size=30,
    targets=["t2_p40_h30", "t2_p50_h30"],
    n_estimators_values=[200, 400, 600],
    max_depth_values=[3, 5, 7],
    learning_rate_values=[0.01, 0.03, 0.1],
    subsample_values=[1.0],
    colsample_bytree_values=[1.0],
    min_child_weight_values=[1.0],
    gamma_values=[0.0],
    reg_alpha_values=[0.0],
    reg_lambda_values=[1.0],
    prob_threshold_long_values=[0.40],
    prob_threshold_short_values=[0.40],
    class_weight="balanced",
    verbose=True,
)

2026-04-23 15:01:55,758 | INFO | No existen métricas previas para model=xgboost | split=valid
2026-04-23 15:01:56,012 | INFO | No existen probabilidades previas para model=xgboost | split=valid
2026-04-23 15:01:56,114 | INFO | Loaded: windows_t2_p40_h30_train.npz
2026-04-23 15:01:56,115 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-23 15:01:56,134 | INFO | Loaded: windows_t2_p40_h30_valid.npz
2026-04-23 15:01:56,134 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-23 15:01:56,155 | INFO | Loaded: windows_t2_p40_h30_test.npz
2026-04-23 15:01:56,155 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-23 15:01:56,158 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-23 15:01:56,159 | INFO | Bundle cargado | target=t2_p40_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | test=(6913, 30, 7)
2026-04-23 15:01:56,237 | INFO | Loaded: windows_t2_p50_h30_train.npz
2026-04-23 15:01:56,237 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)



XGBOOST GRID INCREMENTAL | L=30
targets                    = ['t2_p40_h30', 't2_p50_h30']
n_combinations             = 27
n_estimators_values        = [200, 400, 600]
max_depth_values           = [3, 5, 7]
learning_rate_values       = [0.01, 0.03, 0.1]
subsample_values           = [1.0]
colsample_bytree_values    = [1.0]
min_child_weight_values    = [1.0]
gamma_values               = [0.0]
reg_alpha_values           = [0.0]
reg_lambda_values          = [1.0]
threshold_long_values      = [0.4]
threshold_short_values     = [0.4]
class_weight_mode          = balanced

----------------------------------------------------------------------------------------------------
[1/27] n_estimators=200 | max_depth=3 | learning_rate=0.01 | subsample=1.0 | colsample_bytree=1.0 | min_child_weight=1.0 | gamma=0.0 | reg_alpha=0.0 | reg_lambda=1.0 | thr_long=0.4 | thr_short=0.4

XGBOOST | T2 SEQ2ONE | WINDOW_SIZE=L30
targets            = ['t2_p40_h30', 't2_p50_h30']
class_weight       = balanced
input_mod

2026-04-23 15:01:56,256 | INFO | Loaded: windows_t2_p50_h30_valid.npz
2026-04-23 15:01:56,257 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-23 15:01:56,276 | INFO | Loaded: windows_t2_p50_h30_test.npz
2026-04-23 15:01:56,277 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-23 15:01:56,281 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-23 15:01:56,281 | INFO | Bundle cargado | target=t2_p50_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | test=(6913, 30, 7)



WINDOW_SIZE: 30

TARGET: t2_p40_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

TARGET: t2_p50_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

[EVAL] L30 | split=valid | model=xgboost | class_weight=balanced
  -> L30 | target=t2_p40_h30 | model=xgboost | class_weight=balanced
  -> L30 | target=t2_p50_h30 | model=xgboost | class_weight=balanced


2026-04-23 15:02:00,536 | INFO | Métricas guardadas en: /content/drive/MyDrive/neural_profit/metrics_tuning/classification_metrics/classification_metrics_xgboost_valid.parquet


...

💾 Guardado OK | metrics=52 | prob=357864

----------------------------------------------------------------------------------------------------
[27/27] n_estimators=600 | max_depth=7 | learning_rate=0.1 | subsample=1.0 | colsample_bytree=1.0 | min_child_weight=1.0 | gamma=0.0 | reg_alpha=0.0 | reg_lambda=1.0 | thr_long=0.4 | thr_short=0.4

XGBOOST | T2 SEQ2ONE | WINDOW_SIZE=L30
targets            = ['t2_p40_h30', 't2_p50_h30']
class_weight       = balanced
input_mode         = 2d_flat
n_estimators       = 600
max_depth          = 7
learning_rate      = 0.1
subsample          = 1.0
colsample_bytree   = 1.0
min_child_weight   = 1.0
gamma              = 0.0
reg_alpha          = 0.0
reg_lambda         = 1.0
tree_method        = hist
device             = cuda
n_jobs             = -1
thr_long           = 0.4
thr_short          = 0.4

[BUILD] L30 | n_targets=2


2026-04-23 15:08:00,426 | INFO | Loaded: windows_t2_p50_h30_test.npz
2026-04-23 15:08:00,427 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-23 15:08:00,430 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-23 15:08:00,431 | INFO | Bundle cargado | target=t2_p50_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | test=(6913, 30, 7)



WINDOW_SIZE: 30

TARGET: t2_p40_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

TARGET: t2_p50_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

[EVAL] L30 | split=valid | model=xgboost | class_weight=balanced
  -> L30 | target=t2_p40_h30 | model=xgboost | class_weight=balanced
  -> L30 | target=t2_p50_h30 | model=xgboost | class_weight=balanced

[DONE] L30 | metrics_rows=2 | probabilities_rows=13764


2026-04-23 15:08:39,585 | INFO | Métricas guardadas en: /content/drive/MyDrive/neural_profit/metrics_tuning/classification_metrics/classification_metrics_xgboost_valid.parquet
2026-04-23 15:08:40,212 | INFO | Probabilidades guardadas en: /content/drive/MyDrive/neural_profit/metrics_tuning/classification_probabilities/classification_probabilities_xgboost_valid.parquet


💾 Guardado OK | metrics=54 | prob=371628


## **11.1. Análisis de tuneo grueso**

In [24]:
df_xgb_coarse = results_xgb_coarse["metrics"].copy()

df_xgb_coarse["xgb_config"] = (
    "n_est=" + df_xgb_coarse["n_estimators"].astype(str)
    + " | depth=" + df_xgb_coarse["max_depth"].astype(str)
    + " | lr=" + df_xgb_coarse["learning_rate"].astype(str)
)

summary_xgb_coarse = (
    df_xgb_coarse
    .groupby(
        ["xgb_config", "n_estimators", "max_depth", "learning_rate"],
        as_index=False
    )
    .agg(
        n_targets=("target", "nunique"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        balanced_accuracy_std=("balanced_accuracy", "std"),
        f1_macro_mean=("f1_macro", "mean"),
        f1_macro_std=("f1_macro", "std"),
        accuracy_mean=("accuracy", "mean"),
    )
    .sort_values(
        ["balanced_accuracy_mean", "f1_macro_mean"],
        ascending=False
    )
    .reset_index(drop=True)
)

print("Resumen global por combinación gruesa de XGB")
display(summary_xgb_coarse)

Resumen global por combinación gruesa de XGB


,xgb_config,n_estimators,max_depth,learning_rate,n_targets,balanced_accuracy_mean,balanced_accuracy_std,f1_macro_mean,f1_macro_std,accuracy_mean
0,n_est=600 | depth=3 | lr=0.01,600,3,0.01,2,0.410602,0.000679,0.396930,0.011067,0.403952
1,n_est=400 | depth=3 | lr=0.01,400,3,0.01,2,0.410241,0.002558,0.393472,0.009631,0.402717
2,n_est=200 | depth=3 | lr=0.03,200,3,0.03,2,0.409315,0.002014,0.395747,0.012793,0.402645
3,n_est=400 | depth=3 | lr=0.03,400,3,0.03,2,0.408629,0.000417,0.400025,0.008183,0.403734
4,n_est=200 | depth=3 | lr=0.01,200,3,0.01,2,0.406021,0.001679,0.389303,0.015486,0.398213
5,n_est=200 | depth=5 | lr=0.03,200,5,0.03,2,0.402346,0.007834,0.395109,0.014722,0.398431
6,n_est=600 | depth=3 | lr=0.03,600,3,0.03,2,0.402110,0.005359,0.396467,0.011753,0.398649
7,n_est=400 | depth=5 | lr=0.01,400,5,0.01,2,0.401814,0.004942,0.392091,0.013645,0.396760
8,n_est=600 | depth=5 | lr=0.01,600,5,0.01,2,0.401272,0.004490,0.393725,0.011650,0.397196
9,n_est=200 | depth=3 | lr=0.1,200,3,0.10,2,0.400432,0.005187,0.395418,0.011025,0.397341


Lectura principal

La mejor configuración global identificada en el tuneo grueso es:

```python
n_estimators = 600
max_depth = 3
learning_rate = 0.01
```

Con desempeño:

* balanced_accuracy ≈ 0.4106
* f1_macro ≈ 0.3969

Esta combinación representa el mejor resultado promedio entre los targets evaluados.

Patrón dominante

Se observa una estructura consistente en los resultados:

* max_depth = 3 domina claramente frente a valores mayores
* learning_rate bajo (0.01 – 0.03) presenta mejor desempeño
* un mayor número de estimadores compensa learning_rate bajos

Interpretación:

* el problema favorece modelos simples y regularizados
* evitar árboles profundos (5, 7) es clave
* learning_rate alto (0.1) degrada el desempeño

Relación entre n_estimators y learning_rate

Las mejores configuraciones corresponden a:

* (600, 3, 0.01)
* (400, 3, 0.01)
* (200, 3, 0.03)

Esto confirma que:

* learning_rate bajo requiere más estimadores
* learning_rate alto reduce el número de árboles necesarios, pero empeora la performance

Configuraciones a descartar

Se identifican como subóptimas:

* max_depth = 7
* learning_rate = 0.1
* combinaciones con alta profundidad y alta tasa de aprendizaje

Esto permite reducir significativamente el espacio de búsqueda en el tuneo fino.

Selección para el tuneo fino

No se recomienda elegir una única configuración. Se definen tres configuraciones base:

Configuración principal:

```python
n_estimators = 600
max_depth = 3
learning_rate = 0.01
```

Configuración alternativa:

```python
n_estimators = 400
max_depth = 3
learning_rate = 0.01
```

Configuración secundaria:

```python
n_estimators = 200
max_depth = 3
learning_rate = 0.03
```

Estas configuraciones representan distintas combinaciones dentro de la zona óptima identificada.

Conclusión operativa

El modelo XGBoost presenta mejor desempeño cuando:

* se utilizan árboles poco profundos
* se emplea un learning_rate bajo
* se incrementa el número de estimadores para compensar

La complejidad excesiva introduce sobreajuste y reduce la capacidad de generalización.

El espacio óptimo de hiperparámetros queda claramente delimitado, lo que permite enfocar el tuneo fino de manera eficiente.

Recomendación

Para el tuneo fino se propone:

* fijar max_depth = 3
* trabajar con 2 o 3 combinaciones de n_estimators y learning_rate
* abrir únicamente los siguientes hiperparámetros:

  * subsample
  * colsample_bytree
  * gamma
  * reg_alpha
  * reg_lambda
  * thresholds de probabilidad

Esto evita la explosión combinatoria y mantiene el análisis centrado en la región de mejor desempeño.


# **12. Tuneo fino**

In [ ]:
results_xgb_fine_1 = run_xgboost_grid_incremental(
    window_size=30,
    targets=["t2_p40_h30", "t2_p50_h30"],
    n_estimators_values=[600],
    max_depth_values=[3],
    learning_rate_values=[0.01],
    subsample_values=[0.6, 0.8, 1.0],
    colsample_bytree_values=[0.6, 0.8, 1.0],
    min_child_weight_values=[1.0],
    gamma_values=[0.0, 0.1, 0.3],
    reg_alpha_values=[0.0, 0.1, 1.0],
    reg_lambda_values=[1.0, 10.0, 50.0],
    prob_threshold_long_values=[0.35, 0.40, 0.45, 0.50],
    prob_threshold_short_values=[0.35, 0.40, 0.45, 0.50],
    class_weight="balanced",
    verbose=True,
)

results_xgb_fine_2 = run_xgboost_grid_incremental(
    window_size=30,
    targets=["t2_p40_h30", "t2_p50_h30"],
    n_estimators_values=[400],
    max_depth_values=[3],
    learning_rate_values=[0.01],
    subsample_values=[0.6, 0.8, 1.0],
    colsample_bytree_values=[0.6, 0.8, 1.0],
    min_child_weight_values=[1.0],
    gamma_values=[0.0, 0.1, 0.3],
    reg_alpha_values=[0.0, 0.1, 1.0],
    reg_lambda_values=[1.0, 10.0, 50.0],
    prob_threshold_long_values=[0.35, 0.40, 0.45, 0.50],
    prob_threshold_short_values=[0.35, 0.40, 0.45, 0.50],
    class_weight="balanced",
    verbose=True,
)

results_xgb_fine_3 = run_xgboost_grid_incremental(
    window_size=30,
    targets=["t2_p40_h30", "t2_p50_h30"],
    n_estimators_values=[200],
    max_depth_values=[3],
    learning_rate_values=[0.03],
    subsample_values=[0.6, 0.8, 1.0],
    colsample_bytree_values=[0.6, 0.8, 1.0],
    min_child_weight_values=[1.0],
    gamma_values=[0.0, 0.1, 0.3],
    reg_alpha_values=[0.0, 0.1, 1.0],
    reg_lambda_values=[1.0, 10.0, 50.0],
    prob_threshold_long_values=[0.35, 0.40, 0.45, 0.50],
    prob_threshold_short_values=[0.35, 0.40, 0.45, 0.50],
    class_weight="balanced",
    verbose=True,
)

2026-04-23 15:21:00,510 | INFO | Cargando métricas desde: /content/drive/MyDrive/neural_profit/metrics_tuning/classification_metrics/classification_metrics_xgboost_valid.parquet
2026-04-23 15:21:00,573 | INFO | Cargando probabilidades desde: /content/drive/MyDrive/neural_profit/metrics_tuning/classification_probabilities/classification_probabilities_xgboost_valid.parquet
2026-04-23 15:21:00,751 | INFO | Loaded: windows_t2_p40_h30_train.npz
2026-04-23 15:21:00,752 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-23 15:21:00,771 | INFO | Loaded: windows_t2_p40_h30_valid.npz
2026-04-23 15:21:00,771 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-23 15:21:00,791 | INFO | Loaded: windows_t2_p40_h30_test.npz
2026-04-23 15:21:00,791 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-23 15:21:00,794 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-23 15:21:00,795 | INFO | Bundle cargado | target=t2_p40_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 


XGBOOST GRID INCREMENTAL | L=30
targets                    = ['t2_p40_h30', 't2_p50_h30']
n_combinations             = 3888
n_estimators_values        = [600]
max_depth_values           = [3]
learning_rate_values       = [0.01]
subsample_values           = [0.6, 0.8, 1.0]
colsample_bytree_values    = [0.6, 0.8, 1.0]
min_child_weight_values    = [1.0]
gamma_values               = [0.0, 0.1, 0.3]
reg_alpha_values           = [0.0, 0.1, 1.0]
reg_lambda_values          = [1.0, 10.0, 50.0]
threshold_long_values      = [0.35, 0.4, 0.45, 0.5]
threshold_short_values     = [0.35, 0.4, 0.45, 0.5]
class_weight_mode          = balanced

----------------------------------------------------------------------------------------------------
[1/3888] n_estimators=600 | max_depth=3 | learning_rate=0.01 | subsample=0.6 | colsample_bytree=0.6 | min_child_weight=1.0 | gamma=0.0 | reg_alpha=0.0 | reg_lambda=1.0 | thr_long=0.35 | thr_short=0.35

XGBOOST | T2 SEQ2ONE | WINDOW_SIZE=L30
targets            = ['t

2026-04-23 15:21:00,884 | INFO | Loaded: windows_t2_p50_h30_valid.npz
2026-04-23 15:21:00,884 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-23 15:21:00,904 | INFO | Loaded: windows_t2_p50_h30_test.npz
2026-04-23 15:21:00,905 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-23 15:21:00,908 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-23 15:21:00,908 | INFO | Bundle cargado | target=t2_p50_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | test=(6913, 30, 7)



WINDOW_SIZE: 30

TARGET: t2_p40_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

TARGET: t2_p50_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

[EVAL] L30 | split=valid | model=xgboost | class_weight=balanced
  -> L30 | target=t2_p40_h30 | model=xgboost | class_weight=balanced
  -> L30 | target=t2_p50_h30 | model=xgboost | class_weight=balanced

[DONE] L30 | metrics_rows=2 | probabilities_rows=13764


2026-04-23 15:21:08,175 | INFO | Métricas guardadas en: /content/drive/MyDrive/neural_profit/metrics_tuning/classification_metrics/classification_metrics_xgboost_valid.parquet
2026-04-23 15:21:08,668 | INFO | Probabilidades guardadas en: /content/drive/MyDrive/neural_profit/metrics_tuning/classification_probabilities/classification_probabilities_xgboost_valid.parquet
2026-04-23 15:21:08,739 | INFO | Loaded: windows_t2_p40_h30_train.npz
2026-04-23 15:21:08,739 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-23 15:21:08,757 | INFO | Loaded: windows_t2_p40_h30_valid.npz
2026-04-23 15:21:08,757 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-23 15:21:08,775 | INFO | Loaded: windows_t2_p40_h30_test.npz
2026-04-23 15:21:08,775 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-23 15:21:08,778 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-23 15:21:08,779 | INFO | Bundle cargado | target=t2_p40_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 

...


WINDOW_SIZE: 30

TARGET: t2_p40_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

TARGET: t2_p50_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

[EVAL] L30 | split=valid | model=xgboost | class_weight=balanced
  -> L30 | target=t2_p40_h30 | model=xgboost | class_weight=balanced
  -> L30 | target=t2_p50_h30 | model=xgboost | class_weight=balanced

[DONE] L30 | metrics_rows=2 | probabilities_rows=13764


2026-04-23 15:21:23,548 | INFO | Métricas guardadas en: /content/drive/MyDrive/neural_profit/metrics_tuning/classification_metrics/classification_metrics_xgboost_valid.parquet
2026-04-23 15:21:24,043 | INFO | Probabilidades guardadas en: /content/drive/MyDrive/neural_profit/metrics_tuning/classification_probabilities/classification_probabilities_xgboost_valid.parquet
2026-04-23 15:21:24,113 | INFO | Loaded: windows_t2_p40_h30_train.npz
2026-04-23 15:21:24,113 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-23 15:21:24,131 | INFO | Loaded: windows_t2_p40_h30_valid.npz
2026-04-23 15:21:24,131 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-23 15:21:24,148 | INFO | Loaded: windows_t2_p40_h30_test.npz
2026-04-23 15:21:24,149 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-23 15:21:24,151 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-23 15:21:24,152 | INFO | Bundle cargado | target=t2_p40_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 

💾 Guardado OK | metrics=60 | prob=412920

----------------------------------------------------------------------------------------------------
[4/3888] n_estimators=600 | max_depth=3 | learning_rate=0.01 | subsample=0.6 | colsample_bytree=0.6 | min_child_weight=1.0 | gamma=0.0 | reg_alpha=0.0 | reg_lambda=1.0 | thr_long=0.35 | thr_short=0.5

XGBOOST | T2 SEQ2ONE | WINDOW_SIZE=L30
targets            = ['t2_p40_h30', 't2_p50_h30']
class_weight       = balanced
input_mode         = 2d_flat
n_estimators       = 600
max_depth          = 3
learning_rate      = 0.01
subsample          = 0.6
colsample_bytree   = 0.6
min_child_weight   = 1.0
gamma              = 0.0
reg_alpha          = 0.0
reg_lambda         = 1.0
tree_method        = hist
device             = cuda
n_jobs             = -1
thr_long           = 0.35
thr_short          = 0.5

[BUILD] L30 | n_targets=2


2026-04-23 15:21:24,249 | INFO | Loaded: windows_t2_p50_h30_test.npz
2026-04-23 15:21:24,249 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-23 15:21:24,252 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-23 15:21:24,253 | INFO | Bundle cargado | target=t2_p50_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | test=(6913, 30, 7)



WINDOW_SIZE: 30

TARGET: t2_p40_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

TARGET: t2_p50_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

[EVAL] L30 | split=valid | model=xgboost | class_weight=balanced
  -> L30 | target=t2_p40_h30 | model=xgboost | class_weight=balanced
